In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import polars as pl
import torch
from torch.utils.data import DataLoader, TensorDataset

from fart.model.device import get_device
from fart.model.nbeats import NBeatsNet
from fart.model.nbeats_config import NBeatsConfig
from fart.model.nbeats_dataset import build_return_windows
from fart.model.train_model import prepare_training_data
from fart.utils import get_project_root
from fart.visualization.confidence_calibration import plot_confidence_calibration
from fart.visualization.plot_styles import apply_plot_styles

apply_plot_styles()

In [ ]:
assets_dir = get_project_root() / "assets"
market, interval = "BTC-EUR", "1d"

config = NBeatsConfig()
lookback = config.lookback

X_train, X_test, y_train, y_test = prepare_training_data(
    data_dir=assets_dir,
    market=market,
    interval=interval,
    months=None,
)

n_train = y_train.shape[0]
close_prices = pl.concat([y_train, y_test])
X_all, y_all = build_return_windows(close_prices, lookback)

n_train_windows = max(0, n_train - lookback - 1)
X_train_windows, y_train_windows = X_all[:n_train_windows], y_all[:n_train_windows]
X_test_windows, y_test_windows = X_all[n_train_windows:], y_all[n_train_windows:]

len(close_prices), X_train_windows.shape, X_test_windows.shape